In [1]:
import pandas as pd
import numpy as np
import os

DATA_ROOT = "/kaggle/input/osic-pulmonary-fibrosis-progression"
TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)



In [2]:
from sklearn.preprocessing import LabelEncoder

le_sex = LabelEncoder()
le_smoke = LabelEncoder()
train_df["Sex"] = le_sex.fit_transform(train_df["Sex"])
train_df["SmokingStatus"] = le_smoke.fit_transform(train_df["SmokingStatus"])
test_df["Sex"] = le_sex.transform(test_df["Sex"])
test_df["SmokingStatus"] = le_smoke.transform(test_df["SmokingStatus"])



In [3]:
# Build per‑patient linear models and confidence estimates.
# Confidence is set to max(70, mean absolute residual) – a tighter calibration
# than the previous mean + std approach.
patient_models = {}
patient_confs = {}

for pid, grp in train_df.groupby("Patient"):
    weeks = grp["Weeks"].values
    fvc = grp["FVC"].values
    if len(grp) == 1:
        slope, intercept = 0.0, fvc[0]
        conf = 70.0
    else:
        slope, intercept = np.polyfit(weeks, fvc, 1)
        pred = slope * weeks + intercept
        abs_err = np.abs(pred - fvc)
        conf = max(70.0, np.mean(abs_err))  # tighter confidence
    patient_models[pid] = (slope, intercept)
    patient_confs[pid] = conf

# Global linear trend and its confidence (used for unseen patients)
all_weeks = train_df["Weeks"].values
all_fvc = train_df["FVC"].values
global_slope, global_intercept = np.polyfit(all_weeks, all_fvc, 1)

global_pred = global_slope * all_weeks + global_intercept
global_abs_err = np.abs(global_pred - all_fvc)
global_conf = max(70.0, np.mean(global_abs_err))  # tighter global confidence

# Store baseline week & FVC for each test patient (week 0 measurement)
baseline_info = {}
for pid, grp in test_df.groupby("Patient"):
    idx = np.argmin(np.abs(grp["Weeks"].values))
    week0 = grp.iloc[idx]["Weeks"]
    fvc0 = grp.iloc[idx]["FVC"]
    baseline_info[pid] = (week0, fvc0)



In [4]:
# Generate predictions for every Patient_Week in the sample submission.
pred_fvc = []
pred_conf = []

for pw in sample_sub["Patient_Week"]:
    patient, week_str = pw.rsplit("_", 1)
    week = int(week_str)

    if patient in patient_models:
        slope, intercept = patient_models[patient]
        fvc_pred = slope * week + intercept
        conf = patient_confs[patient]
    else:
        if patient in baseline_info:
            base_week, base_fvc = baseline_info[patient]
            # Anchor the global line to the patient’s baseline measurement
            intercept = base_fvc - global_slope * base_week
            fvc_pred = global_slope * week + intercept
        else:
            fvc_pred = global_slope * week + global_intercept
        conf = global_conf

    fvc_pred = max(0, fvc_pred)  # enforce non‑negative predictions
    pred_fvc.append(fvc_pred)
    pred_conf.append(conf)



In [5]:
# Assemble and write the submission file.
submission = pd.DataFrame(
    {
        "Patient_Week": sample_sub["Patient_Week"],
        "FVC": pred_fvc,
        "Confidence": pred_conf,
    }
)

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}, shape: {submission.shape}")

Submission written to submission.csv, shape: (1908, 3)
